# Notebook 05 - Results Analysis and Paper Assets

**Project:** Explainable Deep Learning for MRI Brain Tumor Classification and Segmentation Using Transfer Learning, MONAI, U-Net, and Grad-CAM

Run notebooks in order. Each notebook writes outputs into the same project folder so later notebooks can reuse them.


## Purpose

This notebook collects outputs from the other notebooks and prepares tables/figures for the final paper.


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    PROJECT_ROOT = Path("/content/brain_tumor_xai_project")
else:
    PROJECT_ROOT = Path.cwd() / "brain_tumor_xai_project"

config_path = PROJECT_ROOT / "project_config.json"
if config_path.exists():
    config = json.loads(config_path.read_text())
    PROJECT_ROOT = Path(config["project_root"])
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    FIGURE_DIR = Path(config["figure_dir"])
else:
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    FIGURE_DIR = PROJECT_ROOT / "figures"
print("Project root:", PROJECT_ROOT)


In [ ]:
cls_path = OUTPUT_DIR / "classification_model_comparison.csv"
if cls_path.exists():
    cls_df = pd.read_csv(cls_path)
else:
    cls_df = pd.DataFrame(columns=["model","accuracy","precision_weighted","recall_weighted","f1_weighted","auc_macro_ovr"])
    print("Classification result not found. Run Notebook 02 first.")
cls_df


In [ ]:
seg_path = OUTPUT_DIR / "segmentation_summary.json"
if seg_path.exists():
    seg_summary = json.loads(seg_path.read_text())
else:
    seg_summary = {"model":"[TO BE FILLED]", "best_val_dice":"[TO BE FILLED]", "roi_size":"[TO BE FILLED]"}
    print("Segmentation result not found. Run Notebook 04 first.")
seg_df = pd.DataFrame([seg_summary])
seg_df


In [ ]:
paper_tables_dir = OUTPUT_DIR / "paper_tables"
paper_tables_dir.mkdir(parents=True, exist_ok=True)
cls_df.to_csv(paper_tables_dir / "classification_results_table.csv", index=False)
seg_df.to_csv(paper_tables_dir / "segmentation_results_table.csv", index=False)
print("Saved paper tables to:", paper_tables_dir)


In [ ]:
figure_files = []
if FIGURE_DIR.exists():
    for ext in ["*.png", "*.jpg", "*.jpeg"]:
        figure_files.extend(FIGURE_DIR.rglob(ext))
fig_df = pd.DataFrame({"figure_path": [str(p) for p in sorted(figure_files)]})
fig_df.to_csv(OUTPUT_DIR / "paper_figure_inventory.csv", index=False)
fig_df.head(30)


In [ ]:
checklist = [
    {"notebook": "01_dataset_preprocessing.ipynb", "required_output": "classification_metadata.csv", "status": (OUTPUT_DIR / "classification_metadata.csv").exists()},
    {"notebook": "02_classification_training.ipynb", "required_output": "classification_model_comparison.csv", "status": (OUTPUT_DIR / "classification_model_comparison.csv").exists()},
    {"notebook": "03_xai_gradcam.ipynb", "required_output": "*_xai_gradcam_summary.csv", "status": len(list(OUTPUT_DIR.glob("*_xai_gradcam_summary.csv"))) > 0},
    {"notebook": "04_monai_segmentation.ipynb", "required_output": "segmentation_summary.json", "status": (OUTPUT_DIR / "segmentation_summary.json").exists()},
]
check_df = pd.DataFrame(checklist)
check_df.to_csv(OUTPUT_DIR / "notebook_execution_checklist.csv", index=False)
check_df


In [ ]:
summary_md = "# Paper Results Summary\n\n"
summary_md += "## Classification Results\n\n"
summary_md += cls_df.to_markdown(index=False) if len(cls_df) else "[TO BE FILLED AFTER RUNNING NOTEBOOK 02]"
summary_md += "\n\n## Segmentation Results\n\n"
summary_md += seg_df.to_markdown(index=False)
summary_md += "\n\n## Figure Inventory\n\n"
summary_md += fig_df.to_markdown(index=False) if len(fig_df) else "[NO FIGURES FOUND YET]"
(OUTPUT_DIR / "paper_results_summary.md").write_text(summary_md)
print(summary_md)


## Send back these outputs for paper finalization

- `outputs/classification_model_comparison.csv`
- `outputs/*_classification_report.csv`
- `outputs/*_confusion_matrix.csv`
- `outputs/*_xai_gradcam_summary.csv`
- `outputs/segmentation_summary.json`
- `outputs/paper_results_summary.md`
- all generated figures from `figures/`
